In [6]:
import pandas as pd

In [ ]:
# load excel dataset
data = pd.read_excel('Data/UNHCR_Flow_Data.xlsx', sheet_name='DATA')

# filter out rows where 'Count' is less than 100
data = data[data['Count'] >= 100].copy()

In [22]:
# adjustable rolling window length (years)
window_size = 3

# aggregate to yearly totals for each origin-asylum pair
pair_yearly = (
    data.groupby(['origin', 'OriginName', 'asylum', 'AsylumName', 'Year'], as_index=False)['Count']
    .sum()
)

# compute rolling windows using continuous calendar years
window_rows = []
for (origin, origin_name, asylum, asylum_name), group in pair_yearly.groupby(['origin', 'OriginName', 'asylum', 'AsylumName']):
    yearly = group.set_index('Year')['Count'].sort_index()
    full_years = range(int(yearly.index.min()), int(yearly.index.max()) + 1)
    yearly = yearly.reindex(full_years, fill_value=0)

    rolling_window = yearly.rolling(window=window_size).sum().dropna()
    for end_year, total in rolling_window.items():
        window_rows.append({
            'origin': origin,
            'OriginName': origin_name,
            'asylum': asylum,
            'AsylumName': asylum_name,
            'start_year': int(end_year) - (window_size - 1),
            'end_year': int(end_year),
            'cumulative_count': float(total),
        })

windows = pd.DataFrame(window_rows)
top_window_per_pair = windows.loc[windows.groupby(['origin', 'asylum'])['cumulative_count'].idxmax()]
top_window_per_pair = top_window_per_pair.sort_values('cumulative_count', ascending=False).reset_index(drop=True)

top_window_per_pair.head(20)

,origin,OriginName,asylum,AsylumName,start_year,end_year,cumulative_count
0,AFG,Afghanistan,PAK,Pakistan,1980,1982,2475440.0
1,SYR,Syrian Arab Rep.,TUR,Türkiye,2013,2015,2437742.0
2,ETH,Ethiopia,SOM,Somalia,1978,1980,1977290.0
3,VEN,Venezuela (Bolivarian Republic of),COL,Colombia,2018,2020,1810339.0
4,AFG,Afghanistan,IRN,Iran (Islamic Rep. of),1980,1982,1670000.0
5,IRQ,Iraq,IRN,Iran (Islamic Rep. of),1989,1991,1432000.0
6,RWA,Rwanda,COD,Dem. Rep. of the Congo,1994,1996,1417138.0
7,UKR,Ukraine,GFR,Germany,2022,2024,1373752.0
8,UKR,Ukraine,POL,Poland,2022,2024,1359992.0
9,UKR,Ukraine,RUS,Russian Federation,2022,2024,1322740.0


In [23]:
import numpy as np

# episode-based conflict significance analysis (variable-length conflicts)
min_activation = 5000           # minimum yearly displaced people to consider a year "active"
activation_multiplier = 2.0     # origin-specific threshold = baseline * multiplier
quiet_years_to_end = 1          # end episode after this many consecutive below-threshold years
duration_weight_alpha = 0.25    # higher values reward longer conflicts more

# aggregate to origin-year totals (sum across all asylum countries)
origin_yearly = (
    data.groupby(['origin', 'OriginName', 'Year'], as_index=False)['Count']
    .sum()
)

def detect_origin_episodes(group):
    series = group.set_index('Year')['Count'].sort_index()
    full_years = range(int(series.index.min()), int(series.index.max()) + 1)
    series = series.reindex(full_years, fill_value=0)

    positive = series[series > 0]
    baseline = float(positive.median()) if not positive.empty else 0.0
    threshold = max(min_activation, baseline * activation_multiplier)

    episodes = []
    in_episode = False
    start_year = None
    low_streak = 0

    for year, value in series.items():
        if not in_episode and value >= threshold:
            in_episode = True
            start_year = int(year)
            low_streak = 0
            continue

        if in_episode:
            if value < threshold:
                low_streak += 1
                if low_streak >= quiet_years_to_end:
                    end_year = int(year) - quiet_years_to_end
                    if end_year >= start_year:
                        episodes.append((start_year, end_year, threshold))
                    in_episode = False
                    start_year = None
                    low_streak = 0
            else:
                low_streak = 0

    if in_episode and start_year is not None:
        episodes.append((start_year, int(series.index.max()), threshold))

    rows = []
    for episode_num, (start, end, threshold_used) in enumerate(episodes, start=1):
        episode_series = series.loc[start:end]
        total_displaced = float(episode_series.sum())
        peak_annual = float(episode_series.max())
        duration_years = int(end - start + 1)
        mean_annual = float(total_displaced / duration_years)
        ramp_up = float(episode_series.diff().fillna(0).max())

        rows.append({
            'origin': group['origin'].iloc[0],
            'OriginName': group['OriginName'].iloc[0],
            'episode': episode_num,
            'start_year': start,
            'end_year': end,
            'duration_years': duration_years,
            'threshold_used': float(threshold_used),
            'total_displaced': total_displaced,
            'peak_annual': peak_annual,
            'mean_annual': mean_annual,
            'ramp_up': ramp_up,
        })

    return rows

episode_rows = []
for _, g in origin_yearly.groupby(['origin', 'OriginName']):
    episode_rows.extend(detect_origin_episodes(g))

episodes_df = pd.DataFrame(episode_rows)
episodes_df['composite_score'] = episodes_df['total_displaced'] * (
    1 + duration_weight_alpha * np.log1p(episodes_df['duration_years'])
)

top_by_total = episodes_df.sort_values('total_displaced', ascending=False).reset_index(drop=True).head(20)
top_by_peak = episodes_df.sort_values('peak_annual', ascending=False).reset_index(drop=True).head(20)
top_by_composite = episodes_df.sort_values('composite_score', ascending=False).reset_index(drop=True).head(20)

print('Top episodes by total displaced:')
display(top_by_total)
print('Top episodes by peak annual displaced:')
display(top_by_peak)
print('Top episodes by balanced composite score:')
display(top_by_composite)

Top episodes by total displaced:


,origin,OriginName,episode,start_year,end_year,duration_years,threshold_used,total_displaced,peak_annual,mean_annual,ramp_up,composite_score
0,SYR,Syrian Arab Rep.,1,2010,2025,16,6928.0,9524520.0,2264590.0,5.952825e+05,1585679.0,1.627077e+07
1,VEN,Venezuela (Bolivarian Republic of),1,2015,2025,11,5000.0,8578012.0,2528237.0,7.798193e+05,2402156.0,1.390690e+07
2,UKR,Ukraine,3,2022,2025,4,10500.0,8125528.0,5781363.0,2.031382e+06,0.0,1.139491e+07
3,AFG,Afghanistan,1,1980,1982,3,273112.0,4177676.0,2354302.0,1.392559e+06,1046116.0,5.625548e+06
4,RWA,Rwanda,6,1994,1996,3,10007.0,2434129.0,2310857.0,8.113763e+05,52272.0,3.277734e+06
5,SUD,Sudan,5,2023,2025,3,67991.0,2396226.0,1069442.0,7.987420e+05,260631.0,3.226695e+06
6,ETH,Ethiopia,1,1978,1980,3,48790.0,2247302.0,1571784.0,7.491007e+05,1135473.0,3.026158e+06
7,AFG,Afghanistan,5,2021,2023,3,273112.0,2170225.0,916238.0,7.234083e+05,0.0,2.922368e+06
8,SRB,Serbia and Kosovo: S/RES/1244 (1999),1,1990,2000,11,32373.0,1901636.0,990211.0,1.728760e+05,858950.0,3.082983e+06
9,SSD,South Sudan,2,2016,2017,2,218167.0,1745075.0,1005508.0,8.725375e+05,265941.0,2.224365e+06


Top episodes by peak annual displaced:


,origin,OriginName,episode,start_year,end_year,duration_years,threshold_used,total_displaced,peak_annual,mean_annual,ramp_up,composite_score
0,UKR,Ukraine,3,2022,2025,4,10500.0,8125528.0,5781363.0,2.031382e+06,0.0,1.139491e+07
1,VEN,Venezuela (Bolivarian Republic of),1,2015,2025,11,5000.0,8578012.0,2528237.0,7.798193e+05,2402156.0,1.390690e+07
2,AFG,Afghanistan,1,1980,1982,3,273112.0,4177676.0,2354302.0,1.392559e+06,1046116.0,5.625548e+06
3,RWA,Rwanda,6,1994,1996,3,10007.0,2434129.0,2310857.0,8.113763e+05,52272.0,3.277734e+06
4,SYR,Syrian Arab Rep.,1,2010,2025,16,6928.0,9524520.0,2264590.0,5.952825e+05,1585679.0,1.627077e+07
5,ETH,Ethiopia,1,1978,1980,3,48790.0,2247302.0,1571784.0,7.491007e+05,1135473.0,3.026158e+06
6,IRQ,Iraq,1,1991,1991,1,63544.0,1422637.0,1422637.0,1.422637e+06,0.0,1.669161e+06
7,SUD,Sudan,5,2023,2025,3,67991.0,2396226.0,1069442.0,7.987420e+05,260631.0,3.226695e+06
8,SSD,South Sudan,2,2016,2017,2,218167.0,1745075.0,1005508.0,8.725375e+05,265941.0,2.224365e+06
9,SRB,Serbia and Kosovo: S/RES/1244 (1999),1,1990,2000,11,32373.0,1901636.0,990211.0,1.728760e+05,858950.0,3.082983e+06


Top episodes by balanced composite score:


,origin,OriginName,episode,start_year,end_year,duration_years,threshold_used,total_displaced,peak_annual,mean_annual,ramp_up,composite_score
0,SYR,Syrian Arab Rep.,1,2010,2025,16,6928.0,9524520.0,2264590.0,5.952825e+05,1585679.0,1.627077e+07
1,VEN,Venezuela (Bolivarian Republic of),1,2015,2025,11,5000.0,8578012.0,2528237.0,7.798193e+05,2402156.0,1.390690e+07
2,UKR,Ukraine,3,2022,2025,4,10500.0,8125528.0,5781363.0,2.031382e+06,0.0,1.139491e+07
3,AFG,Afghanistan,1,1980,1982,3,273112.0,4177676.0,2354302.0,1.392559e+06,1046116.0,5.625548e+06
4,RWA,Rwanda,6,1994,1996,3,10007.0,2434129.0,2310857.0,8.113763e+05,52272.0,3.277734e+06
5,SUD,Sudan,5,2023,2025,3,67991.0,2396226.0,1069442.0,7.987420e+05,260631.0,3.226695e+06
6,SRB,Serbia and Kosovo: S/RES/1244 (1999),1,1990,2000,11,32373.0,1901636.0,990211.0,1.728760e+05,858950.0,3.082983e+06
7,ETH,Ethiopia,1,1978,1980,3,48790.0,2247302.0,1571784.0,7.491007e+05,1135473.0,3.026158e+06
8,AFG,Afghanistan,5,2021,2023,3,273112.0,2170225.0,916238.0,7.234083e+05,0.0,2.922368e+06
9,SSD,South Sudan,2,2016,2017,2,218167.0,1745075.0,1005508.0,8.725375e+05,265941.0,2.224365e+06
